# ============== Dernière version de AINIDSystem =============

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical, Sequence
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split as tts
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import glob
import joblib

2026-03-05 15:16:52.513430: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-05 15:16:53.061306: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-05 15:16:54.973288: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### GPU config

In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

### Load + mapping

In [3]:
all_files = glob.glob('../data/MachineLearningCVE/*.csv')
df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)

In [4]:
# Binary mapping
df['is_attack'] = (df[' Label'] != 'BENIGN').astype(int)

print("Distribution:")
print(df['is_attack'].value_counts())

Distribution:
is_attack
0    2273097
1     557646
Name: count, dtype: int64


### Preproc

In [5]:
X = df.drop(columns=[' Label', 'is_attack']).apply(pd.to_numeric, errors='coerce')
X.replace([np.inf, -np.inf], np.nan, inplace=True)
valid_indices = X.dropna().index
X = X.loc[valid_indices]
y = df['is_attack'].loc[valid_indices].values

In [7]:
column_names = list(X.columns)
joblib.dump(column_names, '../brAIn/memory/expected_columns.pkl')

print(f"Saved {len(column_names)} column names")
print(column_names[:5])

Saved 78 column names
[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets']


### Scale

In [6]:
rs = RobustScaler()
X_scaled = rs.fit_transform(X)

In [18]:
joblib.dump(rs, 'banana_scaler.pkl')

['banana_scaler.pkl']

In [7]:
print(f"Final dataset: {X_scaled.shape}")
print(f"Attack distribution: {np.bincount(y)}")

Final dataset: (2827876, 78)
Attack distribution: [2271320  556556]


### Tts + Class Weight

In [8]:
X_train, X_test, y_train, y_test = tts(
    X_scaled, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train distribution: {np.bincount(y_train)}")

Train: (2262300, 78), Test: (565576, 78)
Train distribution: [1817055  445245]


In [12]:
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print("Class weights:", class_weight_dict)

Class weights: {0: np.float64(0.6225183057199699), 1: np.float64(2.540511403833844)}


### Model

In [13]:
model = Sequential([
    Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    
    Dense(32, activation='relu'),
    Dropout(0.1),
    
    Dense(1, activation='sigmoid') 
])

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)
print(model.summary())

/home/illillillillili/.local/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1772699629.521339   74419 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        20,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65,281 (255.00 KB)

 Trainable params: 64,385 (251.50 KB)

 Non-trainable params: 896 (3.50 KB)

None


In [14]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)
checkpoint = ModelCheckpoint('best_banana_brain.keras', monitor='val_loss', save_best_only=True)

history = model.fit(
    X_train,
    y_train,
    batch_size=512,
    epochs=50,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, lr_scheduler, checkpoint],
    class_weight=class_weight_dict
)

Epoch 1/50


2026-03-05 09:33:58.542135: I external/local_xla/xla/service/service.cc:163] XLA service 0x745778014120 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-05 09:33:58.542164: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-03-05 09:33:58.592977: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-05 09:33:58.872633: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
2026-03-05 09:33:59.058181: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-05 09:33:59.

  33/4419 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - accuracy: 0.6723 - loss: 0.7046 - precision: 0.2857 - recall: 0.4519

I0000 00:00:1772699644.189116   76546 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


4418/4419 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8434 - loss: 0.3904 - precision: 0.5816 - recall: 0.7602

2026-03-05 09:34:23.576510: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-05 09:34:23.576602: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-05 09:34:23.576612: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-05 09:34:23.576619: I external/l

4419/4419 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8435 - loss: 0.3904 - precision: 0.5816 - recall: 0.7602

2026-03-05 09:34:31.344919: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-05 09:34:31.345019: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-05 09:34:31.345030: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-05 09:34:32.091669: I external/l

4419/4419 ━━━━━━━━━━━━━━━━━━━━ 36s 7ms/step - accuracy: 0.9012 - loss: 0.2550 - precision: 0.6989 - recall: 0.8749 - val_accuracy: 0.9666 - val_loss: 0.1472 - val_precision: 0.8871 - val_recall: 0.9513 - learning_rate: 1.0000e-04
Epoch 2/50
4419/4419 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.9633 - loss: 0.1113 - precision: 0.8669 - recall: 0.9612 - val_accuracy: 0.9691 - val_loss: 0.1937 - val_precision: 0.8814 - val_recall: 0.9738 - learning_rate: 1.0000e-04
Epoch 3/50
4419/4419 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9717 - loss: 0.0876 - precision: 0.8935 - recall: 0.9721 - val_accuracy: 0.9493 - val_loss: 0.4619 - val_precision: 0.8023 - val_recall: 0.9853 - learning_rate: 1.0000e-04
Epoch 4/50
4419/4419 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9742 - loss: 0.0791 - precision: 0.9011 - recall: 0.9759 - val_accuracy: 0.9796 - val_loss: 0.2780 - val_precision: 0.9268 - val_recall: 0.9730 - learning_rate: 1.0000e-04
Epoch 5/50
4419/4419 ━━━━━━━━━━━━━━━━━━━━ 15s 3

In [15]:
best_model = tf.keras.models.load_model('best_banana_brain.keras')

y_pred_proba = best_model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

print(classification_report(y_test, y_pred, target_names=['BENIGN', 'ATTACK']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

2026-03-05 09:40:41.165633: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-05 09:40:41.165680: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-05 09:40:41.165707: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-05 09:40:41.917955: I external/l

17675/17675 ━━━━━━━━━━━━━━━━━━━━ 20s 1ms/step
              precision    recall  f1-score   support

      BENIGN       0.99      0.98      0.99    454265
      ATTACK       0.94      0.97      0.96    111311

    accuracy                           0.98    565576
   macro avg       0.97      0.98      0.97    565576
weighted avg       0.98      0.98      0.98    565576


Confusion Matrix:
[[447397   6868]
 [  3299 108012]]
